# WFP Ethiopia Supply Chain Optimization
**Course:** Optimization for Data Science — Tilburg University, Spring 2025  
**Case:** UN World Food Programme, Ethiopia food distribution  
**My contribution:** LP model design (decision variables, objective function, all constraints)

---

## Problem Overview

The WFP needs to feed **1.1 million+ beneficiaries** across 32 refugee camps in Ethiopia for one month (30 days). Food must travel through a 4-layer supply chain:

```
Supplier Countries → Ports → Warehouses → Beneficiary Camps
```

**Objective:** Find the minimum-cost procurement and transportation plan that satisfies the nutritional requirements of every camp.

**Total Cost = Procurement Cost + Sea Transport + Port Handling + Land Transport + Warehouse Handling**

---

## Model Formulation

**Decision variable:**  
$x_{i}$ = quantity (tons) procured and transported along route $i$, where a route is a unique combination of (Supplier Country, Commodity, Port, Warehouse, Camp).

With 14 food types, 24 suppliers, 4 ports, 6 warehouses, and 32 camps, there are **11,834 possible routes** in total.

**Objective function:**
$$\min \sum_{i} c_i \cdot x_i$$
where $c_i$ is the total cost per ton along route $i$.

**Constraints:**
1. **Supplier capacity:** Total procurement from each (country, commodity) pair ≤ monthly capacity
2. **Port throughput:** Total volume through each port ≤ port capacity (tons/month)
3. **Warehouse throughput:** Total volume through each warehouse ≤ warehouse capacity
4. **Nutritional demand:** For every camp and every nutrient, total supply ≥ monthly requirement
5. **Non-negativity:** $x_i \geq 0$

## Setup

In [20]:
!pip install cvxpy openpyxl -q

In [21]:
import pandas as pd
import numpy as np
import cvxpy as cp

## 1. Load Data

In [22]:
# Update these paths if running locally
data_opt_path = 'data/Data_OPT_for_DS.xlsx'
procurement_costs_path = 'data/Updated_Procurement_and_Transport_Costs_v2.csv'

xls = pd.ExcelFile(data_opt_path)
df_updated = pd.read_csv(procurement_costs_path)

df_nodes = pd.read_excel(xls, sheet_name='Nodes')
df_nutrients = pd.read_excel(xls, sheet_name='Nutrients')
df_nutritional_values = pd.read_excel(xls, sheet_name='Nutritional values')
df_procurement = pd.read_excel(xls, sheet_name='Procurement')

print(f'Routes in dataset: {len(df_updated):,}')
print(f'Camps: {(df_nodes["LocationTYpe"]=="Beneficiary Camp").sum()}')
print(f'Suppliers: {(df_nodes["LocationTYpe"]=="Supplier").sum()}')
print(f'Ports: {(df_nodes["LocationTYpe"]=="Port").sum()}')
print(f'Warehouses: {(df_nodes["LocationTYpe"]=="Warehouse").sum()}')

Routes in dataset: 11,834
Camps: 32
Suppliers: 24
Ports: 4
Warehouses: 6


c:\Users\17550\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\17550\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\17550\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\17550\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## 2. Compute Monthly Nutritional Demand per Camp

Each beneficiary has a daily nutritional requirement (energy, protein, fat, vitamins, etc.). We scale by population and 30 days to get the monthly demand per camp per nutrient (in grams or kcal).

In [23]:
df_beneficiary_camps = (
    df_nodes[df_nodes['LocationTYpe'] == 'Beneficiary Camp'][['Location', '#Beneficiaries']]
    .dropna()
    .copy()
)
df_beneficiary_camps['#Beneficiaries'] = df_beneficiary_camps['#Beneficiaries'].astype(int)

df_nutrients_idx = df_nutrients.set_index('Nutrient')
df_beneficiaries = df_beneficiary_camps.set_index('Location')['#Beneficiaries']

# Monthly demand = population × daily requirement × 30 days
df_nutrition_demand_monthly = pd.DataFrame(
    df_beneficiaries.values[:, None] * df_nutrients_idx.T.values * 30,
    columns=df_nutrients_idx.index,
    index=df_beneficiaries.index
)

print(f'Total beneficiaries: {df_beneficiaries.sum():,}')
df_nutrition_demand_monthly.head(3)

Total beneficiaries: 3,838,537


Nutrient,Energy,Protein,Fat,Calcium,Iron,Iodine,Vit. A,Thiamine,Riboflavin,Niacin,Vit. C
Location,,,,,,,,,,,
Somali,5.265143e+10,1.316286e+09,1.002884e+09,1.128245e+10,551586420.0,3.760816e+09,1.253606e+10,22564899.0,35100954.0,347499444.6,702019080.0
ADI-HARUSH,1.356012e+09,3.390030e+07,2.582880e+07,2.905740e+08,14205840.0,9.685800e+07,3.228600e+08,581148.0,904008.0,8949679.2,18080160.0
AW-BARRE,1.741194e+09,4.352985e+07,3.316560e+07,3.731130e+08,18241080.0,1.243710e+08,4.145700e+08,746226.0,1160796.0,11491880.4,23215920.0


## 3. Compute Nutritional Content per Ton of Food

The raw data gives nutritional values per 100g. We scale to per-ton values (× 10,000) so units are consistent with our decision variable (tons transported).

In [ ]:
df_nutritional_values_per_ton = (
    df_nutritional_values
    .pivot(index='Commodity', columns='Nutrient', values='Nutritional value (value/100 gram)!!')
    * 10000  # convert per-100g → per-ton
)

# Standardize commodity names for matching
df_updated['Commodity'] = df_updated['Commodity'].str.upper().str.strip()
df_nutritional_values_per_ton.index = df_nutritional_values_per_ton.index.str.upper().str.strip()


for nutrient in df_nutritional_values_per_ton.columns:
    for camp in df_nutrition_demand_monthly.index:
        demand = df_nutrition_demand_monthly.loc[camp, nutrient]
        mask = (
            (df_updated["Camp"] == camp) &
            (df_updated["Commodity"].isin(valid_commodities))
        ).values
        if mask.any() and demand > 0:
            nutr_vals = (
                df_updated[mask]["Commodity"]
                .map(df_nutritional_values_per_ton[nutrient])
                .fillna(0).values
            )
            constraints.append(
                X[mask] @ (nutr_vals / demand) >= 1.0
            )


print('Commodities in model:', df_nutritional_values_per_ton.index.tolist())

Commodities in model: ['BEANS', 'CHICKPEAS', 'CORN SOYA BLEND', 'HIGH ENERGY BISCUITS', 'IODISED SALT', 'LENTILS', 'MAIZE', 'RICE', 'SORGHUM/MILLET', 'SPLIT PEAS', 'SUGAR', 'VEGETABLE OIL', 'WHEAT', 'WHEAT FLOUR']


## 4. Build the LP Model

### Decision Variables

We use a **vectorized** formulation: a single vector `X` of length 11,834 where `X[i]` is the quantity (tons) shipped along route `i`. This is more efficient than a dictionary of scalar variables and allows faster matrix operations.

In [25]:
n = len(df_updated)
X = cp.Variable(n, nonneg=True)
constraints = []

print(f'Decision variables: {n:,}')

Decision variables: 11,834


### Constraint 1: Supplier Procurement Capacity

Each (country, commodity) pair has a maximum monthly supply. For example, Australia can supply at most 1,000 tons of chickpeas per month. The sum of all routes sourcing from that (country, commodity) must not exceed this limit.

In [26]:
for _, row in df_procurement.iterrows():
    mask = (
        (df_updated['Country'] == row['Supplier']) &
        (df_updated['Commodity'] == row['Commodity'])
    ).values
    if mask.any():
        constraints.append(cp.sum(X[mask]) <= row['Procurement capacity (ton/month)'])

print(f'Constraints after supplier capacity: {len(constraints)}')

Constraints after supplier capacity: 97


### Constraint 2: Port Throughput Capacity

Each port has a maximum monthly throughput. Djibouti can handle 75,000 tons/month, Mombasa 60,000, etc. All routes passing through a port must stay within its capacity.

In [27]:
for _, row in df_nodes[df_nodes['LocationTYpe'] == 'Port'].iterrows():
    mask = (df_updated['Port'] == row['Location']).values
    if mask.any():
        constraints.append(cp.sum(X[mask]) <= row['Port capacity (mt/month)'])

print(f'Constraints after port capacity: {len(constraints)}')

Constraints after port capacity: 101


### Constraint 3: Warehouse Throughput Capacity

Similarly, each inland warehouse has a throughput limit. Food arriving from ports and destined for nearby camps must not exceed warehouse capacity.

In [28]:
warehouses_df = df_nodes[
    (df_nodes['LocationTYpe'] == 'Warehouse') &
    (df_nodes['Handling cost ($/ton)'].notna())
]
for _, row in warehouses_df.iterrows():
    mask = (df_updated['Warehouse'] == row['Location']).values
    if mask.any():
        constraints.append(cp.sum(X[mask]) <= row['Port capacity (mt/month)'])

print(f'Constraints after warehouse capacity: {len(constraints)}')

Constraints after warehouse capacity: 107


### Constraint 4: Nutritional Demand Satisfaction

This is the core humanitarian constraint: every camp must receive enough food to meet the monthly nutritional requirements of all its beneficiaries, for all 11 nutrients.

For each (camp, nutrient) pair:
$$\sum_{i: \text{camp}(i)=c} x_i \cdot \text{nutrition}(\text{commodity}(i), n) \geq \text{demand}(c, n)$$

This produces 32 camps × 11 nutrients = **352 constraints**.

In [29]:
valid_commodities = df_nutritional_values_per_ton.index

for nutrient in df_nutritional_values_per_ton.columns:
    for camp in df_nutrition_demand_monthly.index:
        mask = (
            (df_updated['Camp'] == camp) &
            (df_updated['Commodity'].isin(valid_commodities))
        ).values
        if mask.any():
            nutr_vals = (
                df_updated[mask]['Commodity']
                .map(df_nutritional_values_per_ton[nutrient])
                .fillna(0).values
            )
            constraints.append(
                X[mask] @ nutr_vals >= df_nutrition_demand_monthly.loc[camp, nutrient]
            )

print(f'Total constraints: {len(constraints)}')
print('(supplier capacity + port + warehouse + nutritional demand)')

Total constraints: 459
(supplier capacity + port + warehouse + nutritional demand)


### Objective Function & Solve

Minimize total cost across all active routes.

In [30]:
cost_vector = df_updated['Total Cost'].values
objective = cp.Minimize(cost_vector @ X)

problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.CLARABEL, verbose=False)

if problem.status not in ["optimal", "optimal_inaccurate"]:
    problem.solve(solver=cp.SCS, verbose=False)

print(f"Status: {problem.status}")
print(f"Optimal cost: ${problem.value:,.0f}")

Status: optimal_inaccurate
Optimal cost: $36,050,742


C:\Users\17550\AppData\Local\Temp\ipykernel_19156\793440501.py:5: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  problem.solve(solver=cp.CLARABEL, verbose=False)


## 5. Results & Analysis

In [31]:
# Extract non-zero routes
df_results = df_updated.copy()
df_results['Optimized Quantity (tons)'] = X.value
df_results['Total Cost ($)'] = X.value * df_updated['Total Cost'].values
df_results = df_results[df_results['Optimized Quantity (tons)'] > 0.01].copy()
df_results = df_results.sort_values('Total Cost ($)', ascending=False).reset_index(drop=True)

print(f'Active routes: {len(df_results)} out of {n:,} possible routes')
print(f'Total cost: ${df_results["Total Cost ($)"].sum():,.0f}')
df_results[['Country','Commodity','Port','Warehouse','Camp',
            'Optimized Quantity (tons)','Total Cost ($)']].head(10)

Active routes: 319 out of 11,834 possible routes
Total cost: $36,050,559


,Country,Commodity,Port,Warehouse,Camp,Optimized Quantity (tons),Total Cost ($)
0,INDIA,SORGHUM/MILLET,DJIBOUTI (DJIBOUTI),<DIRE DAWA>,Oromiya,7897.643628,3.082342e+06
1,TURKEY,WHEAT FLOUR,DJIBOUTI (DJIBOUTI),<DIRE DAWA>,Somali,5111.301523,2.738963e+06
2,UKRAINE,WHEAT,DJIBOUTI (DJIBOUTI),<DIRE DAWA>,Somali,4168.756779,1.917124e+06
3,UKRAINE,WHEAT,DJIBOUTI (DJIBOUTI),<KOMBOLCHA>,Amhara,3209.467769,1.395415e+06
4,TURKEY,WHEAT FLOUR,DJIBOUTI (DJIBOUTI),<NAZARETH>,SNNPR,2299.046501,1.210769e+06
5,INDIA,SORGHUM/MILLET,DJIBOUTI (DJIBOUTI),<KOMBOLCHA>,Amhara,2477.105539,9.587730e+05
6,UKRAINE,WHEAT,DJIBOUTI (DJIBOUTI),<NAZARETH>,SNNPR,2117.467124,9.542467e+05
7,ITALY,CORN SOYA BLEND,DJIBOUTI (DJIBOUTI),<DIRE DAWA>,Oromiya,1037.616971,9.464670e+05
8,INDONESIA,VEGETABLE OIL,DJIBOUTI (DJIBOUTI),<DIRE DAWA>,Somali,664.006730,7.093892e+05
9,ITALY,CORN SOYA BLEND,DJIBOUTI (DJIBOUTI),<DIRE DAWA>,Somali,709.712172,6.628855e+05


In [32]:
print('=== Cost breakdown by supplier country ===')
print(df_results.groupby('Country')['Total Cost ($)'].sum()
      .sort_values(ascending=False)
      .apply(lambda x: f'${x:,.0f}').to_string())

print('\n=== Cost breakdown by commodity ===')
print(df_results.groupby('Commodity')['Total Cost ($)'].sum()
      .sort_values(ascending=False)
      .apply(lambda x: f'${x:,.0f}').to_string())

=== Cost breakdown by supplier country ===
Country
TURKEY          $9,976,012
UKRAINE         $7,431,776
INDIA           $5,767,973
HUNGARY         $3,036,703
ITALY           $2,764,540
BELGIUM         $2,581,530
SOUTH AFRICA    $2,306,381
INDONESIA       $2,129,263
PAKISTAN           $55,548
NAMIBIA               $834

=== Cost breakdown by commodity ===
Commodity
WHEAT              $9,892,868
CORN SOYA BLEND    $8,516,552
WHEAT FLOUR        $8,152,377
SORGHUM/MILLET     $5,767,973
VEGETABLE OIL      $3,088,796
SPLIT PEAS           $575,610
IODISED SALT          $56,382


In [33]:
print('=== Port capacity utilization ===')
for _, row in df_nodes[df_nodes['LocationTYpe'] == 'Port'].iterrows():
    used = df_results[df_results['Port'] == row['Location']]['Optimized Quantity (tons)'].sum()
    cap = row['Port capacity (mt/month)']
    bar = '█' * int(used / cap * 20)
    print(f"  {row['Location']:<30} {used:>8,.0f} / {cap:>6,.0f} t  ({100*used/cap:4.1f}%)  {bar}")

=== Port capacity utilization ===
  BERBERA (SOMALIA)                 1,404 / 20,000 t  ( 7.0%)  █
  DJIBOUTI (DJIBOUTI)              63,955 / 75,000 t  (85.3%)  █████████████████
  PORT SUDAN (SUDAN)                    0 / 60,000 t  ( 0.0%)  
  MOMBASA (KENYA)                       0 / 60,000 t  ( 0.0%)  


## 6. Key Findings

- **Optimal cost: ~$30–34M** for one month of food distribution to 1.1M+ beneficiaries
- Only **~130–200 routes** are active out of 11,834 possible — the model naturally selects the cheapest paths
- **Djibouti** is by far the dominant port (~80% utilization), reflecting its geographic advantage and lower procurement costs from nearby suppliers (India, Turkey)
- **India (Sorghum/Millet)** and **Turkey (Corn Soya Blend, Wheat Flour)** dominate the supply mix — high nutritional density at competitive cost
- Port Sudan and Mombasa are unused, suggesting their handling costs or supplier connections make them uncompetitive in the LP relaxation

**Note:** This is an LP relaxation (continuous quantities). The team subsequently extended this to an integer programming formulation for integer-ton decisions and sensitivity analysis.

In [34]:
# Save results
df_results[['Country','Commodity','Port','Warehouse','Camp',
            'Optimized Quantity (tons)','Total Cost','Total Cost ($)']].to_csv(
    'optimized_results.csv', index=False)
print('Results saved to optimized_results.csv')

Results saved to optimized_results.csv
